<a href="https://colab.research.google.com/github/nitha604/news-article-categorisation/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers torch scikit-learn nltk spacy wordcloud streamlit -q
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("fancyzhx/ag_news")  # downloads directly, no issues on Colab

train_df = ds['train'].to_pandas()
test_df  = ds['test'].to_pandas()
df = pd.concat([train_df, test_df], ignore_index=True)

# Filter + rename
df = df[df['label'].isin([1, 3])].copy()
df['category'] = df['label'].map({1: 'Sports', 3: 'Technology'})
df.drop(columns=['label'], inplace=True)

# Balance — 1000 per class
df = df.groupby('category').sample(n=1000, random_state=42).reset_index(drop=True)
print(df['category'].value_counts())

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

category
Sports        1000
Technology    1000
Name: count, dtype: int64


In [4]:
# Try to connect to Google Drive; fall back to local storage if it fails
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/news_project/'
    os.makedirs(SAVE_PATH, exist_ok=True)
    print("Drive mounted successfully!")
except Exception as e:
    print(f"Could not mount Drive due to: {e}")
    print("Falling back to local Colab session storage instead...")
    SAVE_PATH = './'

# Save your dataframe
file_path = os.path.join(SAVE_PATH, 'ag_news_filtered.csv')
df.to_csv(file_path, index=False)
print(f"Saved to {file_path} ")

Mounted at /content/drive
Drive mounted successfully!
Saved to /content/drive/MyDrive/news_project/ag_news_filtered.csv ✅


In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print(" Loading AG News dataset from HuggingFace...")

# Load the dataset (train + test splits)
dataset = load_dataset("fancyzhx/ag_news")

# Convert each split to a pandas DataFrame
train_df = dataset['train'].to_pandas()
test_df  = dataset['test'].to_pandas()

# Combine train and test into one big DataFrame
df = pd.concat([train_df, test_df], ignore_index=True)

print(f"\n Dataset loaded!")
print(f" Total shape: {df.shape}  (rows, columns)")
print(f"\n Label distribution:")
print(df['label'].value_counts().sort_index())
print("\n(0=World, 1=Sports, 2=Business, 3=Sci/Tech)")


 Loading AG News dataset from HuggingFace...

 Dataset loaded!
 Total shape: (127600, 2)  (rows, columns)

 Label distribution:
label
0    31900
1    31900
2    31900
3    31900
Name: count, dtype: int64

(0=World, 1=Sports, 2=Business, 3=Sci/Tech)


In [6]:
# Filter: keep only Sports (1) and Sci/Tech (3)
df_filtered = df[df['label'].isin([1, 3])].copy()

# Map numeric labels to meaningful category names
label_map = {
    1: 'Sports',
    3: 'Technology'
}
df_filtered['category'] = df_filtered['label'].map(label_map)

# Drop the original numeric label column — we don't need it anymore
df_filtered.drop(columns=['label'], inplace=True)

print(f" Filtered dataset shape: {df_filtered.shape}")
print(f"\n Category counts:")
print(df_filtered['category'].value_counts())
print()
print(df_filtered.head(3))

 Filtered dataset shape: (63800, 2)

 Category counts:
category
Technology    31900
Sports        31900
Name: count, dtype: int64

                                                 text    category
78  'Madden,' 'ESPN' Football Score in Different W...  Technology
79  Group to Propose New High-Speed Wireless Forma...  Technology
80  AOL to Sell Cheap PCs to Minorities and Senior...  Technology


In [7]:
# Sample 1000 rows per category (for balance)
SAMPLES_PER_CLASS = 1000

df_balanced = (
    df_filtered
    .groupby('category', group_keys=False)
    .apply(lambda x: x.sample(n=SAMPLES_PER_CLASS, random_state=42))
)

# Reset the index so row numbers are clean and sequential
df_balanced = df_balanced.reset_index(drop=True)

print(f" Balanced dataset shape: {df_balanced.shape}")
print(f"\n Final category counts:")
print(df_balanced['category'].value_counts())
print()
print("Sample rows:")
print(df_balanced.sample(3, random_state=1)[['text', 'category']])


 Balanced dataset shape: (2000, 2)

 Final category counts:
category
Sports        1000
Technology    1000
Name: count, dtype: int64

Sample rows:
                                                   text    category
674   ASHADO WINS FOR PLETCHER Horseplayers started ...      Sports
1699  Florida Judge: Recount Votes A state rule barr...  Technology
1282  Paychex Inc. Founder Golisano Steps Down (AP) ...  Technology


/tmp/ipykernel_2074/3565812841.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=SAMPLES_PER_CLASS, random_state=42))


In [8]:
# --- Education placeholder headlines (Indian context) ---
education_headlines = [
    "CBSE announces changes to Class 12 board exam pattern for 2024",
    "IIT Bombay launches new BTech program in Artificial Intelligence",
    "JEE Advanced 2024 registration begins on NTA official website",
    "UGC directs universities to implement NEP 2020 curriculum by next year",
    "NEET UG results declared; topper from Tamil Nadu scores perfect 720",
    "Government announces free coaching for SC/ST students for UPSC exam",
    "Delhi University introduces four-year undergraduate programme",
    "IIM Ahmedabad reports highest placement package of Rs 1.2 crore",
    "Mid-day meal scheme to be expanded to secondary schools across India",
    "NCERT revises Class 10 textbooks to include digital literacy chapter",
    "Study abroad applications from India surge 40% post-pandemic",
    "PM eVIDYA portal crosses 10 crore registrations milestone",
    "Maharashtra to make coding compulsory from Class 6 in state schools",
    "NIT Trichy introduces minor degree in Data Science for all branches",
    "Students protest fee hike at Jadavpur University in Kolkata",
    "GATE 2024 results out; CSE paper sees record number of aspirants",
    "Karnataka govt announces scholarship for girl students in rural areas",
    "India ranks 40th in global education index released by UNESCO",
    "Private schools in Bengaluru adopt hybrid learning model permanently",
    "IISc Bangalore researchers develop AI tool to personalise learning"
]

# --- Politics placeholder headlines (Indian context) ---
politics_headlines = [
    "Parliament session adjourned amid opposition uproar over Manipur issue",
    "BJP announces candidates for upcoming Rajasthan assembly elections",
    "PM Modi holds bilateral talks with US President on trade and defence",
    "Congress launches 'Bharat Jodo Yatra 2.0' ahead of general elections",
    "Supreme Court to hear petitions challenging electoral bonds scheme",
    "AAP wins municipal elections in Gujarat; BJP concedes defeat",
    "Union Cabinet approves Women's Reservation Bill in Lok Sabha",
    "NDA allies meet in Delhi to discuss seat sharing for 2024 elections",
    "Rahul Gandhi addresses rally in Telangana ahead of state polls",
    "Chief Minister of Maharashtra resigns amid political crisis",
    "India abstains from UN vote on Russia-Ukraine ceasefire resolution",
    "President Droupadi Murmu inaugurates new Parliament building",
    "Election Commission announces model code of conduct for five states",
    "GST Council meets to decide on inclusion of petroleum products",
    "Shiv Sena split: Supreme Court to deliver verdict on party symbol row",
    "Bihar political crisis deepens as JDU withdraws support from NDA",
    "India and China agree to resume normal patrolling at border areas",
    "Finance Minister presents Union Budget 2026 in Lok Sabha",
    "Trinamool Congress wins by-elections in all five West Bengal seats",
    "CAA implementation: Centre notifies rules after four-year delay"
]

# Create DataFrames for each
df_education = pd.DataFrame({
    'text': education_headlines,
    'category': 'Education'
})

df_politics = pd.DataFrame({
    'text': politics_headlines,
    'category': 'Politics'
})

# Concatenate everything together!
df_final = pd.concat([df_balanced, df_education, df_politics], ignore_index=True)

print(f" Final dataset shape: {df_final.shape}")
print(f"\n Final category distribution:")
print(df_final['category'].value_counts())


 Final dataset shape: (2040, 2)

 Final category distribution:
category
Sports        1000
Technology    1000
Education       20
Politics        20
Name: count, dtype: int64


In [9]:
# Save the final DataFrame to Google Drive as a CSV
save_path = '/content/drive/MyDrive/news_project/raw_data.csv'

df_final.to_csv(save_path, index=False)

print(f" Data saved successfully!")
print(f" File location: {save_path}")
print(f" Total rows saved: {len(df_final)}")
print(f"\n Preview of saved data (first 5 rows):")
print(df_final.head(5).to_string(index=True))


 Data saved successfully!
 File location: /content/drive/MyDrive/news_project/raw_data.csv
 Total rows saved: 2040

 Preview of saved data (first 5 rows):
                                                                                                                                                                                                                                                                                                      text category
0                                                                      Barrera outslugs Morales for title Somehow Marco Antonio Barrera and Erik Morales duplicated, and perhaps even surpassed, the action and drama of their all-time great first battle, which took place four years ago at 122 pounds.   Sports
1                                                                                                             Ajax finally comes to terms with Trabelsi Amsterdam, the Netherlands (Sports Network) - Ajax defender Hatem Trabelsi fi